In [2]:
import torch
# 1. Make a fake Sentinel-2 patch: patch = torch.rand(13, 64, 64) (13 bands, 64×64 pixels). Print its shape and dtype.
patch = torch.rand(13, 64, 64)
print(patch.shape, patch.dtype)

torch.Size([13, 64, 64]) torch.float32


In [3]:
# 2. Take out the green band (B3) and the NIR band (B8). In a 13-band stack (B1, B2, B3, B4, B5, B6, B7, B8, B8A, …), which index is each one? Careful, Python counts from 0.
# green band (B3)
green_band = patch[2, :, :]
# NIR band (B8)
nir_band = patch[7, :, :]
print(green_band.shape, nir_band.shape)

torch.Size([64, 64]) torch.Size([64, 64])


In [ ]:
# 3. Compute NDWI for every pixel in one line, with no loops. What shape does it have?
# ⚠️ A hidden bug for real data. What happens to ndwi if a pixel has green + nir = 0? Real Sentinel-2 scenes have nodata pixels = 0 at the edges of the image. Find a way to protect against this. Hint: a tiny number has a name, "epsilon".
#ndwi = (green_band - nir_band) / (green_band + nir_band)
ndwi = (green_band - nir_band) / (green_band + nir_band + 1e-10)  # add epsilon to avoid division by zero
print(ndwi.shape)

torch.Size([64, 64])


In [5]:
# 4. Make a water mask: NDWI > 0. What dtype does it have? How many pixels are "water"?
water_mask = ndwi > 0
print(water_mask.dtype, water_mask.sum())

torch.bool tensor(2070)


In [6]:
# 5. Compute the mean of each band (the result should have 13 numbers). Hint: .mean(dim=...). Which dims do you average over?
mean_per_band = patch.mean(dim=(1, 2))
print(mean_per_band.shape, mean_per_band)

torch.Size([13]) tensor([0.4982, 0.5025, 0.5064, 0.5038, 0.5001, 0.5001, 0.4922, 0.4997, 0.5022,
        0.5014, 0.4951, 0.5026, 0.4921])


In [7]:
# 6. Stack 4 patches into a batch. What shape do you get? Write down what each number means.
batch = torch.stack([patch, patch, patch, patch], dim=0)
print(batch.shape)  # (4, 13, 64, 64) -> (batch_size, num_bands, height, width)

torch.Size([4, 13, 64, 64])


In [11]:
# normalise patch per band ((patch - mean) / std), using keepdim=True. Check that each band now has a mean of about 0 and a std of about 1.
normalized_patch = (patch - patch.mean(dim=(1,2), keepdim=True))  / patch.std(dim=(1, 2), keepdim=True)
print(normalized_patch.mean(dim=(1, 2)), normalized_patch.std(dim=(1, 2)))

tensor([-3.1665e-08, -2.4447e-08,  3.8650e-08,  1.3784e-07, -3.2887e-08,
        -6.2282e-09, -4.2841e-08,  9.8720e-08, -7.9861e-08, -3.4925e-08,
         1.0338e-07,  7.2760e-08,  8.5216e-08]) tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000])
